In [2]:
# !pip install nflreadpy   # uncomment if the package hasn't already been installed

In [3]:
import joblib
import numpy as np
import pandas as pd
import nflreadpy as nfl
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
import gc
import polars as pl
import nflreadpy as nfl
from nflreadpy.config import update_config
update_config(cache_mode="off")   # don't keep the raw download in RAM

start_season, end_season = 2006, 2026   # fewer seasons = less memory; widen if you have the RAM

# don't load all 370 columns, only the ones we actually need:
columns = [
    "season", "game_id", "play_type", "posteam", "home_team", "result",
    "game_seconds_remaining", "half_seconds_remaining", "down", "ydstogo",
    "yardline_100", "score_differential", "posteam_timeouts_remaining",
    "defteam_timeouts_remaining", "wp", "total_home_score", "total_away_score",
]
keep_plays = ["run", "pass", "punt", "field_goal", "qb_kneel", "qb_spike"]

In [5]:
# load one season at a time and trim it immediately so that the peak memory stays low instead of approaching max capacity early on in runtime
frames = []
for season in range(start_season, end_season + 1):
    frames.append(
        nfl.load_pbp(seasons=[season])
        .select(columns)
        .filter(pl.col("play_type").is_in(keep_plays))
    )

pbp_pandas = pl.concat(frames).to_pandas()
del frames
gc.collect()
#print(pbp_pandas.shape, round(pbp_pandas.memory_usage(deep=True).sum() / 1e6), "MB")

48

In [6]:
# true = all seasons; if faster run needed, use something like range(2006, 2027)
pbp_pandas = nfl.load_pbp(seasons=True).to_pandas()
print(pbp_pandas.shape)

(1298029, 372)


In [7]:
play_types = ["run", "pass", "punt", "field_goal"]

df = pbp_pandas[
    pbp_pandas["play_type"].isin(play_types)
    & pbp_pandas["posteam"].notna() #posteam means "team possessing the ball"
    & pbp_pandas["result"].notna()
    & (pbp_pandas["result"] != 0)
].copy()
#possessing team is the home team; boolean
df["posteam_is_home"] = (df["posteam"] == df["home_team"]).astype(int) 
#result column is margin of victory from perspective of home team. change the perspective to away team by flipping the sign if the possessing team is not the home team
pos_result = np.where(df["posteam_is_home"] == 1, df["result"], -df["result"]) 
#get whether the possessing team wins depending on whether the result, i.e. final score margin, is positive
df["posteam_win"] = (pos_result > 0).astype(int)
#converts play type column to fixed categories so that gradient boosting function can use it
df["play_type"] = pd.Categorical(df["play_type"].astype(str), categories=play_types)

features = [
    "game_seconds_remaining",
    "half_seconds_remaining",
    "down",
    "ydstogo",
    "yardline_100",
    "score_differential",
    "posteam_timeouts_remaining",
    "defteam_timeouts_remaining",
    "posteam_is_home",
]
features_play = features + ["play_type"]

required = ["posteam_win", "down", "ydstogo", "yardline_100", "score_differential", "game_seconds_remaining"]
model_df = df.dropna(subset=required)
train = model_df[model_df["season"] <= 2024]
test = model_df[model_df["season"] >= 2025]
y_train, y_test = train["posteam_win"], test["posteam_win"]
print(len(train), "train rows /", len(test), "test rows")
print(train["play_type"].value_counts())

960615 train rows / 51486 test rows
play_type
pass          501180
run           368978
punt           63923
field_goal     26534
Name: count, dtype: int64


comparing brier score loss on logistic vs gradient boosting vs adding play type to the latter

In [8]:
"""LR_NUM = [c for c in STATE if c != "down"]


def make_logistic():
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), LR_NUM),
        ("down_ohe", OneHotEncoder(handle_unknown="ignore"), ["down"]),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])
"""
def make_gb():
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=100,
        l2_regularization=1.0, early_stopping=False,
        categorical_features="from_dtype",   # picks up the `play_type` category column
        random_state=42,
    )

"""lr_state = make_logistic().fit(train[features], y_train)
gb_state = make_gb().fit(train[features], y_train)"""
gb_play = make_gb().fit(train[features_play], y_train)

In [9]:
"""def metrics(name, y, p):
    return {"model": name, "brier": brier_score_loss(y, p), "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "auc": roc_auc_score(y, p)}

p_lr = lr_state.predict_proba(test[STATE])[:, 1]
p_gb_state = gb_state.predict_proba(test[STATE])[:, 1]
p_gb_play = gb_play.predict_proba(test[FEATURES_PLAY])[:, 1]

pd.DataFrame([
    metrics("logistic, state only", y_test, p_lr),
    metrics("boosting, state only", y_test, p_gb_state),
    metrics("boosting, state + play type", y_test, p_gb_play),
]).round(4)"""

'def metrics(name, y, p):\n    return {"model": name, "brier": brier_score_loss(y, p), "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), "auc": roc_auc_score(y, p)}\n\np_lr = lr_state.predict_proba(test[STATE])[:, 1]\np_gb_state = gb_state.predict_proba(test[STATE])[:, 1]\np_gb_play = gb_play.predict_proba(test[FEATURES_PLAY])[:, 1]\n\npd.DataFrame([\n    metrics("logistic, state only", y_test, p_lr),\n    metrics("boosting, state only", y_test, p_gb_state),\n    metrics("boosting, state + play type", y_test, p_gb_play),\n]).round(4)'

interpretation: adding play type to the gradient boosting algorithm does not significantly skew the results. gradient boosting is a bit better than logistic which is good because one of its most important benefits is it can run on cpu even for a large dataset, which is what we need within the scope of this projext

## win probaibility depending on next play
keep game state fixed and swap the play for each iteration; return all results. plays are limited to sensible options. run/pass on downs 1-3; punt/field goal only available on 4th down.

In [10]:
def predict_wp_by_play(model, game_state):
    #game_state: dict with every key in STATE. return possession-team win probability for each allowed play.
    allowed = play_types if game_state["down"] == 4 else ["run", "pass"]
    rows = pd.DataFrame([game_state] * len(allowed))
    rows["play_type"] = pd.Categorical(allowed, categories=play_types)
    probs = model.predict_proba(rows[features_play])[:, 1]
    return pd.Series(probs, index=allowed, name="posteam_wp")

#test based on a few different game states
second_and_six = {
    "game_seconds_remaining": 600, "half_seconds_remaining": 600, "down": 2, "ydstogo": 6,
    "yardline_100": 45, "score_differential": 2,
    "posteam_timeouts_remaining": 3, "defteam_timeouts_remaining": 3, "posteam_is_home": 1,
}
fourth_and_two = {
    "game_seconds_remaining": 600, "half_seconds_remaining": 600, "down": 4, "ydstogo": 2,
    "yardline_100": 35, "score_differential": -3,
    "posteam_timeouts_remaining": 2, "defteam_timeouts_remaining": 3, "posteam_is_home": 1,
}
print(predict_wp_by_play(gb_play, second_and_six).round(4), "\n")
print(predict_wp_by_play(gb_play, fourth_and_two).round(4))

run     0.7571
pass    0.7620
Name: posteam_wp, dtype: float64 

run           0.4201
pass          0.4123
punt          0.4093
field_goal    0.4155
Name: posteam_wp, dtype: float64


In [11]:
joblib.dump(gb_play, "wp_play_model.joblib")

['wp_play_model.joblib']

In [12]:
# !pip install requests   # uncomment if it isn't installed
import json
import time
import requests
from IPython.display import clear_output, display

ESPN_SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
HEADERS = {"User-Agent": "Mozilla/5.0 (class project)"}

def _get_scoreboard():
    r = requests.get(ESPN_SCOREBOARD, headers=HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

def list_games():
    """show games that are on the current scoreboard. state is either 'pre', 'in' (live) or 'post'."""
    rows = []
    for ev in _get_scoreboard().get("events", []):
        t = ev.get("status", {}).get("type", {})
        rows.append({"event_id": ev.get("id"), "game": ev.get("shortName") or ev.get("name"),
                     "state": t.get("state"), "status": t.get("shortDetail")})
    return pd.DataFrame(rows)

def _find_event(data, event_id):
    ev = next((e for e in data.get("events", []) if str(e.get("id")) == str(event_id)), None)
    if ev is None:
        raise ValueError(f"event_id {event_id} is not on the current scoreboard. Run list_games().")
    return ev

def debug_raw_situation(event_id):
    """print the raw status and situation JSON so you can confirm the field names used below."""
    ev = _find_event(_get_scoreboard(), event_id)
    comp = ev["competitions"][0]
    print(json.dumps({"status": ev.get("status"), "situation": comp.get("situation")}, indent=2)[:4000])

In [13]:
def _clock_seconds(status):
    try:
        return float(status.get("clock"))
    except (TypeError, ValueError):
        pass
    try:
        m, s = str(status.get("displayClock", "0:00")).split(":")
        return int(m) * 60 + float(s)
    except Exception:
        return 0.0

def _time_remaining(period, clock):
    """returns (game_seconds_remaining, half_seconds_remaining). overtime uses the OT clock."""
    period = int(period)
    if period >= 5:
        return clock, clock
    game = (4 - period) * 900 + clock
    half = ((2 - period) if period <= 2 else (4 - period)) * 900 + clock
    return game, half

def _yardline_100(possession_text, pos_abbr):
    """possessionText like 'PHI 45' means the ball is on PHI's 45 yd line. convert to yards from the
    end zone the possession team is attacking. midfield is returned as is."""
    if not possession_text:
        return None
    parts = str(possession_text).split()
    try:
        n = int(parts[-1])
    except ValueError:
        return None
    if len(parts) == 1:
        return n
    return 100 - n if parts[0] == pos_abbr else n

def get_live_state(event_id):
    """fetch one game and map it to the model's dictionary keys.
    returns {"state": dict or None, "meta": dict, "problems": [str]}."""
    ev = _find_event(_get_scoreboard(), event_id)
    comp = ev["competitions"][0]
    status = ev.get("status", comp.get("status", {}))
    sides = {c["homeAway"]: c for c in comp["competitors"]}
    home, away = sides["home"], sides["away"]
    sit = comp.get("situation") or {}
    problems = []

    meta = {
        "game": ev.get("shortName") or ev.get("name"),
        "status": status.get("type", {}).get("shortDetail"),
        "game_state": status.get("type", {}).get("state"),
        "home": home["team"].get("abbreviation"), "away": away["team"].get("abbreviation"),
        "home_score": int(home.get("score", 0)), "away_score": int(away.get("score", 0)),
        "possession_text": sit.get("possessionText"),
        "down_distance_text": sit.get("downDistanceText") or sit.get("shortDownDistanceText"),
    }

    if meta["game_state"] != "in":
        problems.append(f"game is not live (state = {meta['game_state']!r}).")
        return {"state": None, "meta": meta, "problems": problems}

    poss_id = str(sit.get("possession")) if sit.get("possession") is not None else None
    if poss_id == str(home["team"].get("id")):
        pos, dfn = home, away
    elif poss_id == str(away["team"].get("id")):
        pos, dfn = away, home
    else:
        problems.append("no team has possession in the feed right now (between plays, kickoff or timeout).")
        return {"state": None, "meta": meta, "problems": problems}

    down, dist = sit.get("down"), sit.get("distance")
    if down not in (1, 2, 3, 4) or dist is None:
        problems.append(f"no valid down/distance in the feed (down={down!r}, distance={dist!r}).")
        return {"state": None, "meta": meta, "problems": problems}

    yl = _yardline_100(sit.get("possessionText"), pos["team"].get("abbreviation"))
    if yl is None:
        problems.append("could not read the ball position from possessionText.")
        return {"state": None, "meta": meta, "problems": problems}

    clock = _clock_seconds(status)
    game_s, half_s = _time_remaining(status.get("period", 1), clock)

    to = {"home": sit.get("homeTimeouts"), "away": sit.get("awayTimeouts")}
    pos_side = "home" if pos is home else "away"
    def_side = "away" if pos is home else "home"
    for side in (pos_side, def_side):
        if to[side] is None:
            problems.append(f"{side} timeouts missing from the feed; assuming 3.")
            to[side] = 3

    state = {
        "game_seconds_remaining": game_s,
        "half_seconds_remaining": half_s,
        "down": int(down),
        "ydstogo": int(dist),
        "yardline_100": int(yl),
        "score_differential": int(pos.get("score", 0)) - int(dfn.get("score", 0)),
        "posteam_timeouts_remaining": int(to[pos_side]),
        "defteam_timeouts_remaining": int(to[def_side]),
        "posteam_is_home": 1 if pos is home else 0,
    }
    meta["posteam"] = pos["team"].get("abbreviation")
    return {"state": state, "meta": meta, "problems": problems}

In [14]:
def show_live(model, event_id):
    """Fetch the live state, run the model, and print a readable summary."""
    snap = get_live_state(event_id)
    m, state = snap["meta"], snap["state"]
    print(f"{m['game']}  |  {m['away']} {m['away_score']} - {m['home_score']} {m['home']}  |  {m['status']}")
    for p in snap["problems"]:
        print("note:", p)
    if state is None:
        print("No prediction right now.")
        return snap, None

    print(f"{m['posteam']} ball: {m['down_distance_text']} | ball on {m['possession_text']} "
          f"(yardline_100 = {state['yardline_100']})")
    probs = predict_wp_by_play(model, state)
    table = pd.DataFrame({
        f"{m['posteam']} win prob": probs,
        f"{m['home']} (home) win prob": probs if state["posteam_is_home"] else 1 - probs,
    }).round(3)
    table.index.name = "play called"
    display(table)
    return snap, table

def watch_game(model, event_id, every=20, max_updates=90):
    """Refresh the prediction every `every` seconds (minimum 15). Interrupt the kernel to stop."""
    every = max(every, 15)
    try:
        for _ in range(max_updates):
            clear_output(wait=True)
            try:
                show_live(model, event_id)
            except Exception as exc:
                print("fetch failed, will retry:", exc)
            print(f"\n(updated {time.strftime('%H:%M:%S')}, refreshing every {every}s; stop the cell to quit)")
            time.sleep(every)
    except KeyboardInterrupt:
        print("stopped.")

## how to use
1. call `list_games()` and copy an `event_id` for a game whose `state` is `in`.
2. during that game, run `debug_raw_situation(event_id)` once and check the keys (`down`, `distance`, `possession`, `possessionText`, `homeTimeouts`, `awayTimeouts`).
3. call `show_live(gb_play, event_id)` for a one-off prediction, or `watch_game(gb_play, event_id)` to auto-refresh.

    sample usage below:

In [16]:
list_games()

,event_id,game,state,status
0,401873022,KC @ SEA,in,11:22 - 1st
1,401873011,PIT VS NO,post,Final
2,401873017,SF @ ATL,post,Final
3,401873013,CLE @ TEN,post,Final
4,401873014,IND @ MIN,post,Final
5,401873015,MIA @ NYJ,post,Final
6,401873018,TB @ CAR,post,Final
7,401873012,CIN @ BAL,post,Final
8,401873016,NYG @ HOU,post,Final/OT
9,401873019,DEN @ ARI,post,Final


In [ ]:
EVENT_ID = 	401873022 #replace this number after calling list_games
debug_raw_situation(EVENT_ID)
show_live(gb_play, EVENT_ID)
watch_game(gb_play, EVENT_ID, every=20)

KC @ SEA  |  KC 0 - 0 SEA  |  7:12 - 1st
KC ball: 1st & 10 at KC 10 | ball on KC 10 (yardline_100 = 90)


,KC win prob,SEA (home) win prob
play called,,
run,0.436,0.564
pass,0.455,0.545



(updated 20:37:06, refreshing every 20s; stop the cell to quit)
